In [ ]:
import pickle
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

# ── 1) Load pkl payload (true per-participant segment embeddings + all labels) ─────
with open("deep-prep-ai-audio-embeddings.pkl", "rb") as f:
    payload = pickle.load(f)

emb_list = payload["transcript_embeddings"]  # list of (n_segments, embed_dim)

LABEL_COLS = [
    "interview_score",
    "personality_score",
    "answer_score",
    "speaking_skills",
    "agreeableness",
    "conscientiousness",
    "neuroticism",
    "openness",
    "confidence_score",
]

Y = np.stack([np.array(payload[c], dtype=np.float32) for c in LABEL_COLS], axis=1)  # (N, num_targets)

N = len(emb_list)
if N == 0:
    raise ValueError("Empty transcript_embeddings list.")

MAX_SEQ_LEN = 64
embed_dim = np.array(emb_list[0], dtype=np.float32).shape[1]
num_targets = Y.shape[1]

# X: (N, MAX_SEQ_LEN, embed_dim)
X = np.zeros((N, MAX_SEQ_LEN, embed_dim), dtype=np.float32)
# key_padding_mask: True for PAD positions
key_padding_mask = np.ones((N, MAX_SEQ_LEN), dtype=bool)

for i, arr in enumerate(emb_list):
    arr = np.array(arr, dtype=np.float32)
    if arr.ndim != 2:
        continue
    n = min(arr.shape[0], MAX_SEQ_LEN)
    if n > 0:
        X[i, :n, :] = arr[:n, :]
        key_padding_mask[i, :n] = False

# Keep only rows with complete targets
valid_rows = np.isfinite(Y).all(axis=1)
X = X[valid_rows]
key_padding_mask = key_padding_mask[valid_rows]
Y = Y[valid_rows]

idx = np.arange(len(Y))
train_idx, val_idx = train_test_split(idx, test_size=0.2, random_state=42)

X_train, X_val = X[train_idx], X[val_idx]
mask_train, mask_val = key_padding_mask[train_idx], key_padding_mask[val_idx]
y_train, y_val = Y[train_idx], Y[val_idx]

# ── 2) Dataset ────────────────────────────────────────────────────────────────
class EmbeddingSeqDataset(Dataset):
    def __init__(self, X, key_padding_mask, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.key_padding_mask = torch.tensor(key_padding_mask, dtype=torch.bool)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.key_padding_mask[idx], self.y[idx]

train_ds = EmbeddingSeqDataset(X_train, mask_train, y_train)
val_ds = EmbeddingSeqDataset(X_val, mask_val, y_val)

BATCH_SIZE = 16
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_dl = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

# ── 3) Model: Transformer encoder over segment embeddings ─────────────────────
class TransformerSeqRegressor(nn.Module):
    def __init__(
        self,
        input_dim,
        max_seq_len,
        num_targets,
        d_model=128,
        nhead=4,
        num_layers=2,
        dropout=0.1,
    ):
        super().__init__()

        self.input_proj = nn.Linear(input_dim, d_model)

        # Learnable [CLS] token
        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))

        # Learnable positional embeddings for (CLS + tokens)
        self.pos_embed = nn.Parameter(torch.zeros(1, max_seq_len + 1, d_model))

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=d_model * 4,
            dropout=dropout,
            batch_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        self.head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, 64),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(64, num_targets),
        )

    def forward(self, x, key_padding_mask):
        # x: (batch, seq_len, input_dim)
        # key_padding_mask: (batch, seq_len) with True at PAD positions
        batch_size, seq_len, _ = x.shape

        x = self.input_proj(x)  # (batch, seq_len, d_model)

        cls = self.cls_token.expand(batch_size, 1, -1)  # (batch, 1, d_model)
        x = torch.cat([cls, x], dim=1)  # (batch, seq_len+1, d_model)

        x = x + self.pos_embed[:, : seq_len + 1, :]

        # Never mask CLS token
        cls_mask = torch.zeros((batch_size, 1), dtype=torch.bool, device=x.device)
        full_mask = torch.cat([cls_mask, key_padding_mask], dim=1)  # (batch, seq_len+1)

        x = self.encoder(x, src_key_padding_mask=full_mask)
        cls_out = x[:, 0, :]  # (batch, d_model)
        return self.head(cls_out)  # (batch, num_targets)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = TransformerSeqRegressor(
    input_dim=embed_dim,
    max_seq_len=MAX_SEQ_LEN,
    num_targets=num_targets,
).to(device)
print(model)
print("Targets:", LABEL_COLS)

# ── 4) Training loop ───────────────────────────────────────────────────────────
LR = 1e-4
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=2, min_lr=1e-7
)
loss_fn = nn.MSELoss()
GRAD_CLIP_NORM = 1.0


def run_epoch(dl, train=True):
    model.train(train)
    total_loss = 0.0

    with torch.set_grad_enabled(train):
        for X_batch, mask_batch, y_batch in dl:
            X_batch = X_batch.to(device)
            mask_batch = mask_batch.to(device)
            y_batch = y_batch.to(device)

            preds = model(X_batch, mask_batch)
            loss = loss_fn(preds, y_batch)

            if train:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
                optimizer.step()

            total_loss += loss.item() * len(y_batch)

    return total_loss / len(dl.dataset)

EPOCHS = 30
MIN_EPOCHS_BEFORE_STOP = 15
PATIENCE = 8
MIN_DELTA = 1e-4

best_val_loss = float("inf")
best_epoch = -1
epochs_no_improve = 0
best_state = None

for epoch in range(EPOCHS):
    train_loss = run_epoch(train_dl, train=True)
    val_loss = run_epoch(val_dl, train=False)
    scheduler.step(val_loss)

    if val_loss < best_val_loss - MIN_DELTA:
        best_val_loss = val_loss
        best_epoch = epoch + 1
        epochs_no_improve = 0
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    elif epoch + 1 >= MIN_EPOCHS_BEFORE_STOP:
        epochs_no_improve += 1

    if (epoch + 1) % 5 == 0:
        lr_now = optimizer.param_groups[0]["lr"]
        print(
            f"Epoch {epoch+1:3d} | train MSE: {train_loss:.4f} | val MSE: {val_loss:.4f} | lr: {lr_now:.2e}"
        )

    if epoch + 1 >= MIN_EPOCHS_BEFORE_STOP and epochs_no_improve >= PATIENCE:
        print(
            f"Early stopping at epoch {epoch+1}. Best val MSE {best_val_loss:.4f} at epoch {best_epoch}."
        )
        break

if best_state is not None:
    device = next(model.parameters()).device
    model.load_state_dict({k: v.to(device) for k, v in best_state.items()})

TransformerSeqRegressor(
  (input_proj): Linear(in_features=384, out_features=128, bias=True)
  (encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
        )
        (linear1): Linear(in_features=128, out_features=512, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=512, out_features=128, bias=True)
        (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (head): Sequential(
    (0): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
    (1): Linear(in_features=128, out_features=64, bias=True)
    (2): GELU(approximate='none')
    (3): Dropout(p=0.